# 📊 Module 18 — NumPy, Pandas & Matplotlib (Data Handling)

## 🎯 What you will learn
1. **NumPy**: arrays, dtypes, shapes, indexing, slicing, boolean masks, broadcasting, vectorisation, statistics, linear algebra
2. **Pandas**: Series, DataFrame, reading CSV/JSON, selection (`loc`/`iloc`), filtering, sorting, missing data, `groupby`, merge/join, pivot, apply, dates
3. **Matplotlib**: line, bar, scatter, histogram (with install instructions)
4. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐ (data science electives) — and a superpower for real projects.

> Install once: `pip install numpy pandas matplotlib` (NumPy & pandas are already installed on this machine; Matplotlib still needs installing for Part C).

---
# PART A — NumPy

NumPy provides the **`ndarray`**: a fixed-type, contiguous-memory, N-dimensional array. Operations are **vectorised** (run in fast C) — typically 10–100× faster than Python loops.

## 1. Creating arrays

In [1]:
import numpy as np

a = np.array([1, 2, 3, 4])
m = np.array([[1, 2, 3], [4, 5, 6]])
print(a, a.dtype, a.shape, a.ndim, a.size)
print(m, m.shape, m.ndim)

print(np.zeros((2, 3)))
print(np.ones(3, dtype=int))
print(np.full((2, 2), 7))
print(np.eye(3))
print(np.arange(0, 10, 2))            # like range
print(np.linspace(0, 1, 5))           # 5 evenly spaced points, endpoints included
print(np.random.default_rng(42).integers(1, 10, size=(2, 3)))
print(np.empty(2).shape, np.zeros_like(m), np.arange(6).reshape(2, 3))

[1 2 3 4] int64 (4,) 1 4
[[1 2 3]
 [4 5 6]] (2, 3) 2
[[0. 0. 0.]
 [0. 0. 0.]]
[1 1 1]
[[7 7]
 [7 7]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[1 7 6]
 [4 4 8]]
(2,) [[0 0 0]
 [0 0 0]] [[0 1 2]
 [3 4 5]]


## 2. Data types (dtype)
Arrays hold **one** type: `int8/16/32/64`, `uint8`, `float32/64`, `bool`, `complex128`, `str_`, `datetime64`.

In [2]:
x = np.array([1, 2, 3], dtype=np.float32)
print(x.dtype, x.itemsize, x.nbytes)
print(np.array([1, 2.5, 3]).dtype)           # upcasting to float
print(np.array([1, "a"]).dtype)              # becomes strings
print(np.array([1.9, 2.1]).astype(int))      # astype truncates
print(np.iinfo(np.int8).max, np.array([127], dtype=np.int8) + 1)   # integer overflow wraps around!

float32 4 12
float64
<U21
[1 2]
127 [-128]


## 3. Indexing & slicing

In [3]:
a = np.arange(10, 20)
print(a[0], a[-1], a[2:5], a[::2], a[::-1])

M = np.arange(1, 13).reshape(3, 4)
print(M)
print(M[1, 2], M[1][2])            # element
print(M[0], M[:, 0])               # row 0, column 0
print(M[1:, 1:3])                  # sub-matrix
print(M[[0, 2], [1, 3]])           # fancy indexing (pairs): (0,1) and (2,3)
print(M[:, [0, 3]])

# Slices are VIEWS, not copies!
view = M[0, :2]
view[0] = 999
print(M[0])
safe = M[0, :2].copy()

10 19 [12 13 14] [10 12 14 16 18] [19 18 17 16 15 14 13 12 11 10]
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
7 7
[1 2 3 4] [1 5 9]
[[ 6  7]
 [10 11]]
[ 2 12]
[[ 1  4]
 [ 5  8]
 [ 9 12]]
[999   2   3   4]


## 4. Boolean masks & filtering

In [4]:
data = np.array([15, 22, 8, 31, 4, 27])
mask = data > 10
print(mask, data[mask])
print(data[(data > 5) & (data < 30)])           # & | ~ instead of and/or/not (parentheses required)
data2 = data.copy(); data2[data2 < 10] = 0
print(data2)
print(np.where(data > 20, "high", "low"))
print(np.count_nonzero(data > 10), (data > 10).sum(), np.any(data > 30), np.all(data > 3))

[ True  True False  True False  True] [15 22 31 27]
[15 22  8 27]
[15 22  0 31  0 27]
['low' 'high' 'low' 'high' 'low' 'high']
4 4 True True


## 5. Vectorised maths & broadcasting

In [5]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print(a + b, a * b, b / a, a ** 2, np.sqrt(a), np.exp(a).round(2))
print(a * 10, a + 1)                              # scalar broadcast to every element

# Broadcasting: shapes are compared from the right; size-1 dimensions stretch
col = np.array([[0], [10], [20]])                # shape (3,1)
row = np.array([1, 2, 3])                        # shape (3,)
print(col + row)                                 # shape (3,3)

# Speed test: Python loop vs vectorised
import time
n = 1_000_000
lst = list(range(n)); arr = np.arange(n)
t = time.perf_counter(); s1 = sum(x * 2 for x in lst); py = time.perf_counter() - t
t = time.perf_counter(); s2 = (arr * 2).sum();        npt = time.perf_counter() - t
print(s1 == s2, f"python {py:.4f}s  numpy {npt:.4f}s  -> {py / npt:.0f}x faster")

[11 22 33] [10 40 90] [10. 10. 10.] [1 4 9] [1.         1.41421356 1.73205081] [ 2.72  7.39 20.09]
[10 20 30] [2 3 4]
[[ 1  2  3]
 [11 12 13]
 [21 22 23]]
True python 0.1135s  numpy 0.0074s  -> 15x faster


## 6. Reshaping, stacking, aggregating

In [6]:
a = np.arange(12)
print(a.reshape(3, 4).shape, a.reshape(2, -1).shape)          # -1 = infer
M = a.reshape(3, 4)
print(M.T.shape, M.flatten().shape, M.ravel().shape)
print(np.vstack([M[0], M[1]]), np.hstack([[1, 2], [3]]))
print(np.concatenate([M, M], axis=0).shape, np.concatenate([M, M], axis=1).shape)
print(np.split(np.arange(6), 3))

scores = np.array([[70, 80, 90], [60, 85, 75], [95, 65, 88]])
print(scores.sum(), scores.mean(), scores.max(), scores.min(), scores.std().round(2))
print("per column:", scores.mean(axis=0))     # axis=0 collapses rows -> one value per column
print("per row   :", scores.mean(axis=1))
print(scores.argmax(), scores.argmax(axis=1), np.sort(scores, axis=1)[:, ::-1])
print(np.median(scores), np.percentile(scores, 90), scores.cumsum(axis=1))

(3, 4) (2, 6)
(4, 3) (12,) (12,)
[[0 1 2 3]
 [4 5 6 7]] [1 2 3]
(6, 4) (3, 8)
[array([0, 1]), array([2, 3]), array([4, 5])]
708 78.66666666666667 95 60 11.29
per column: [75.         76.66666667 84.33333333]
per row   : [80.         73.33333333 82.66666667]
6 [2 1 0] [[90 80 70]
 [85 75 60]
 [95 88 65]]
80.0 91.0 [[ 70 150 240]
 [ 60 145 220]
 [ 95 160 248]]


## 7. Linear algebra & random numbers

In [7]:
A = np.array([[2, 1], [1, 3]]); B = np.array([[1, 0], [4, 5]])
print(A @ B)                                   # matrix multiply (also np.dot)
print(A * B)                                   # element-wise!
print(np.linalg.det(A).round(2), np.linalg.inv(A).round(3))
print(np.linalg.solve(A, np.array([3, 5])))    # solve  2x + y = 3, x + 3y = 5
vals, vecs = np.linalg.eig(A); print(vals.round(3))

rng = np.random.default_rng(seed=0)            # modern reproducible generator
print(rng.random(3).round(3), rng.integers(0, 100, 5), rng.normal(0, 1, 3).round(2))
print(rng.choice(["a", "b", "c"], 5), rng.permutation(5))

[[ 6  5]
 [13 15]]
[[ 2  0]
 [ 4 15]]
5.0 [[ 0.6 -0.2]
 [-0.2  0.4]]
[0.8 1.4]
[1.382 3.618]
[0.637 0.27  0.041] [ 7  1 17 81 64] [ 1.3   0.95 -0.7 ]
['c' 'b' 'c' 'a' 'c'] [2 1 3 4 0]


---
# PART B — Pandas

Pandas gives you **Series** (labelled 1-D) and **DataFrame** (labelled 2-D table) — Excel/SQL power inside Python.

## 8. Series

In [8]:
import pandas as pd
s = pd.Series([90, 85, 70], index=["Asha", "Ravi", "Meena"], name="marks")
print(s)
print(s["Ravi"], s.iloc[0], s[s > 80].index.tolist(), s.mean(), s.idxmax())
print(s * 1.1)

Asha     90
Ravi     85
Meena    70
Name: marks, dtype: int64
85 90 ['Asha', 'Ravi'] 81.66666666666667 Asha
Asha     99.0
Ravi     93.5
Meena    77.0
Name: marks, dtype: float64


## 9. DataFrame — creating and inspecting

In [9]:
df = pd.DataFrame({
    "name":   ["Asha", "Ravi", "Meena", "Tom", "Kiran", "Sara"],
    "dept":   ["IT", "HR", "IT", "Sales", "IT", "HR"],
    "age":    [25, 32, 28, 45, 38, None],
    "salary": [50000, 42000, 61000, 39000, 72000, 45000],
    "joined": pd.to_datetime(["2021-03-01", "2019-07-15", "2020-01-10", "2015-11-20", "2018-05-05", "2022-08-30"]),
})
print(df)
print(df.shape, list(df.columns), df.dtypes.to_dict(), sep="\n")
print(df.head(2)); print(df.tail(2))
print(df.describe().round(1))
df.info()

    name   dept   age  salary     joined
0   Asha     IT  25.0   50000 2021-03-01
1   Ravi     HR  32.0   42000 2019-07-15
2  Meena     IT  28.0   61000 2020-01-10
3    Tom  Sales  45.0   39000 2015-11-20
4  Kiran     IT  38.0   72000 2018-05-05
5   Sara     HR   NaN   45000 2022-08-30
(6, 5)
['name', 'dept', 'age', 'salary', 'joined']
{'name': <StringDtype(na_value=nan)>, 'dept': <StringDtype(na_value=nan)>, 'age': dtype('float64'), 'salary': dtype('int64'), 'joined': dtype('<M8[us]')}
   name dept   age  salary     joined
0  Asha   IT  25.0   50000 2021-03-01
1  Ravi   HR  32.0   42000 2019-07-15
    name dept   age  salary     joined
4  Kiran   IT  38.0   72000 2018-05-05
5   Sara   HR   NaN   45000 2022-08-30
        age   salary               joined
count   5.0      6.0                    6
mean   33.6  51500.0  2019-08-08 12:00:00
min    25.0  39000.0  2015-11-20 00:00:00
25%    28.0  42750.0  2018-08-22 00:00:00
50%    32.0  47500.0  2019-10-12 12:00:00
75%    38.0  58250.0  202

## 10. Selecting data

| Goal | Code |
|---|---|
| one column (Series) | `df["salary"]` |
| several columns | `df[["name", "salary"]]` |
| rows by **label** | `df.loc[row_labels, col_labels]` |
| rows by **position** | `df.iloc[row_pos, col_pos]` |
| filter rows | `df[df["salary"] > 50000]` |

In [10]:
print(df["name"].tolist())
print(df[["name", "salary"]].head(3))
print(df.loc[0, "name"], df.iloc[1, 2], df.loc[1:2, ["name", "dept"]], sep="\n")
print(df[df["dept"] == "IT"])
print(df[(df["salary"] > 45000) & (df["dept"].isin(["IT", "HR"]))][["name", "salary"]])   # & | ~ with parentheses
print(df.query("age > 30 and salary < 60000")[["name"]])
print(df[df["name"].str.startswith("A")])

['Asha', 'Ravi', 'Meena', 'Tom', 'Kiran', 'Sara']
    name  salary
0   Asha   50000
1   Ravi   42000
2  Meena   61000
Asha
32.0
    name dept
1   Ravi   HR
2  Meena   IT
    name dept   age  salary     joined
0   Asha   IT  25.0   50000 2021-03-01
2  Meena   IT  28.0   61000 2020-01-10
4  Kiran   IT  38.0   72000 2018-05-05
    name  salary
0   Asha   50000
2  Meena   61000
4  Kiran   72000
   name
1  Ravi
3   Tom
   name dept   age  salary     joined
0  Asha   IT  25.0   50000 2021-03-01


## 11. Adding, modifying, sorting

In [11]:
df["bonus"] = df["salary"] * 0.1
df["grade"] = np.where(df["salary"] >= 50000, "A", "B")
df["tenure"] = (pd.Timestamp("2025-01-01") - df["joined"]).dt.days // 365
df = df.rename(columns={"dept": "department"})
print(df[["name", "salary", "bonus", "grade", "tenure"]])
print(df.sort_values(["department", "salary"], ascending=[True, False])[["name", "department", "salary"]])
print(df.nlargest(2, "salary")[["name", "salary"]])
df = df.drop(columns=["bonus"])

    name  salary   bonus grade  tenure
0   Asha   50000  5000.0     A       3
1   Ravi   42000  4200.0     B       5
2  Meena   61000  6100.0     A       4
3    Tom   39000  3900.0     B       9
4  Kiran   72000  7200.0     A       6
5   Sara   45000  4500.0     B       2
    name department  salary
5   Sara         HR   45000
1   Ravi         HR   42000
4  Kiran         IT   72000
2  Meena         IT   61000
0   Asha         IT   50000
3    Tom      Sales   39000
    name  salary
4  Kiran   72000
2  Meena   61000


## 12. Missing data

In [12]:
print(df["age"].isna().sum(), df.isna().sum().to_dict())
print(df["age"].fillna(df["age"].mean()).tolist())
print(df.dropna(subset=["age"]).shape)
clean = df.assign(age=df["age"].fillna(df["age"].median()))
print(clean["age"].tolist())

1 {'name': 0, 'department': 0, 'age': 1, 'salary': 0, 'joined': 0, 'grade': 0, 'tenure': 0}
[25.0, 32.0, 28.0, 45.0, 38.0, 33.6]
(5, 7)
[25.0, 32.0, 28.0, 45.0, 38.0, 32.0]


## 13. GroupBy — split, apply, combine

In [13]:
g = df.groupby("department")
print(g["salary"].mean())
print(g["salary"].agg(["count", "mean", "max"]))
print(df.groupby("department").agg(avg_salary=("salary", "mean"), headcount=("name", "count"), oldest=("age", "max")))
print(df.groupby(["department", "grade"]).size())
print(df["department"].value_counts())
print(df.groupby("department")["salary"].transform(lambda s: s - s.mean()).round(0).tolist())

department
HR       43500.0
IT       61000.0
Sales    39000.0
Name: salary, dtype: float64
            count     mean    max
department                       
HR              2  43500.0  45000
IT              3  61000.0  72000
Sales           1  39000.0  39000
            avg_salary  headcount  oldest
department                               
HR             43500.0          2    32.0
IT             61000.0          3    38.0
Sales          39000.0          1    45.0
department  grade
HR          B        2
IT          A        3
Sales       B        1
dtype: int64
department
IT       3
HR       2
Sales    1
Name: count, dtype: int64
[-11000.0, -1500.0, 0.0, 0.0, 11000.0, 1500.0]


## 14. Merging / joining / concatenating

In [14]:
depts = pd.DataFrame({"department": ["IT", "HR", "Finance"], "floor": [3, 1, 2]})
print(pd.merge(df[["name", "department"]], depts, on="department", how="left"))     # like SQL LEFT JOIN
print(pd.merge(df[["name", "department"]], depts, on="department", how="inner").shape)
print(pd.concat([df.head(1), df.tail(1)])[["name"]])

    name department  floor
0   Asha         IT    3.0
1   Ravi         HR    1.0
2  Meena         IT    3.0
3    Tom      Sales    NaN
4  Kiran         IT    3.0
5   Sara         HR    1.0
(5, 3)
   name
0  Asha
5  Sara


## 15. Pivot tables, apply, dates, strings

In [15]:
print(pd.pivot_table(df, values="salary", index="department", columns="grade", aggfunc="mean", fill_value=0))
print(df["salary"].apply(lambda x: f"₹{x:,}").head(3).tolist())
print(df["name"].str.upper().tolist(), df["name"].str.len().tolist())
print(df["joined"].dt.year.tolist(), df["joined"].dt.day_name().tolist()[:2])
print(df.set_index("name").loc["Ravi", "salary"])
print(df.sort_values("joined").iloc[0]["name"], df["salary"].corr(df["age"]).round(3))

# Time series resampling
ts = pd.Series(range(6), index=pd.date_range("2025-01-01", periods=6, freq="D"))
print(ts.resample("2D").sum().tolist(), ts.rolling(3).mean().round(2).tolist())

grade             A        B
department                  
HR              0.0  43500.0
IT          61000.0      0.0
Sales           0.0  39000.0
['₹50,000', '₹42,000', '₹61,000']
['ASHA', 'RAVI', 'MEENA', 'TOM', 'KIRAN', 'SARA'] [4, 4, 5, 3, 5, 4]
[2021, 2019, 2020, 2015, 2018, 2022] ['Monday', 'Monday']
42000
Tom -0.176
[1, 5, 9] [nan, nan, 1.0, 2.0, 3.0, 4.0]


## 16. Reading & writing files

In [16]:
df.to_csv("employees.csv", index=False)
back = pd.read_csv("employees.csv", parse_dates=["joined"])
print(back.shape, back["joined"].dtype)
df.to_json("employees.json", orient="records", date_format="iso")
print(pd.read_json("employees.json").shape)
# Others: read_excel/to_excel (needs openpyxl), read_sql, read_html, read_parquet
print(pd.read_csv(pd.io.common.StringIO("a,b\n1,2\n3,4")).sum().to_dict())

(6, 7) datetime64[us]
(6, 7)
{'a': 4, 'b': 6}


---
# PART C — Matplotlib (plotting)

Install with `pip install matplotlib`. In notebooks add `%matplotlib inline`. The plot cell below ships without saved output (Matplotlib wasn't installed when this notebook was built) — install it, run the cell, and the charts appear.

In [ ]:
# notest
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Matplotlib is not installed. Run:  pip install matplotlib   then re-run this cell.')
else:

    fig, axes = plt.subplots(2, 2, figsize=(10, 7))

    x = np.linspace(0, 2 * np.pi, 100)
    axes[0, 0].plot(x, np.sin(x), label="sin"); axes[0, 0].plot(x, np.cos(x), "--", label="cos")
    axes[0, 0].set(title="Line plot", xlabel="x", ylabel="y"); axes[0, 0].legend(); axes[0, 0].grid(True)

    axes[0, 1].bar(["IT", "HR", "Sales"], [3, 2, 1], color=["tab:blue", "tab:orange", "tab:green"])
    axes[0, 1].set_title("Bar chart")

    rng = np.random.default_rng(1)
    axes[1, 0].scatter(rng.normal(size=100), rng.normal(size=100), alpha=0.6)
    axes[1, 0].set_title("Scatter")

    axes[1, 1].hist(rng.normal(size=1000), bins=25, edgecolor="black")
    axes[1, 1].set_title("Histogram")

    plt.tight_layout()
    plt.show()

    # Pandas has built-in plotting: df.groupby("department")["salary"].mean().plot(kind="bar")

---
# 🕵️ Hidden Facts & Pro Tricks

In [17]:
# 1. Boolean masks + sum() = counting; mean() = percentage
print((df["salary"] > 45000).mean() * 100, "% earn > 45k")

# 2. df.style / display options / method chaining for readable pipelines
result = (df
          .dropna(subset=["age"])
          .assign(age_group=lambda d: pd.cut(d["age"], [0, 30, 40, 100], labels=["<30", "30-40", "40+"]))
          .groupby("age_group", observed=True)["salary"].mean()
          .round(0))
print(result)

# 3. np.where is a vectorised if/else; np.select for multiple conditions
marks = np.array([95, 72, 55, 30])
print(np.select([marks >= 90, marks >= 60, marks >= 40], ["A", "B", "C"], default="F"))

# 4. Copy vs view: NumPy slices are views; fancy/boolean indexing returns copies
# 5. Avoid loops over DataFrame rows (iterrows is slow); use vectorised ops or .apply
# 6. Memory savings: downcast numeric dtypes / use category dtype for repeated strings
big = pd.Series(["IT", "HR"] * 50_000)
print(big.memory_usage(deep=True), big.astype("category").memory_usage(deep=True))

# 7. Chained assignment df[df.x>1]["y"] = 0 does NOT work (SettingWithCopy). Use df.loc[df.x>1, "y"] = 0
# 8. pd.to_numeric(errors="coerce") turns junk into NaN; pd.to_datetime handles dozens of formats
print(pd.to_numeric(pd.Series(["1", "2", "oops"]), errors="coerce").tolist())

# 9. df.duplicated() / drop_duplicates(); df.sample(frac=.2); df.corr(numeric_only=True)
print(df.duplicated().sum())

# 10. np.unique(return_counts=True) is a fast Counter for arrays
print(np.unique([3, 1, 3, 2, 3], return_counts=True))

# 11. np.set_printoptions(suppress=True, precision=3) to make output readable
# 12. Convert between worlds: df.to_numpy(), df.to_dict("records"), pd.DataFrame(list_of_dicts)
print(pd.DataFrame([{"a": 1, "b": 2}, {"a": 3, "b": 4}]).to_dict("records"))

# 13. NaN is a float: an int column with a missing value becomes float64 (use "Int64" nullable dtype)
print(pd.Series([1, None]).dtype, pd.Series([1, None], dtype="Int64").dtype)

# 14. eval/query strings & numexpr speed up big expressions; polars is a newer, faster DataFrame library
# 15. np.nan == np.nan is False -> use np.isnan / pd.isna
print(np.nan == np.nan, np.isnan(np.nan))

50.0 % earn > 45k
age_group
<30      55500.0
30-40    57000.0
40+      39000.0
Name: salary, dtype: float64
['A' 'B' 'C' 'F']
1000132 100153
[1.0, 2.0, nan]
0
(array([1, 2, 3]), array([1, 1, 3]))
[{'a': 1, 'b': 2}, {'a': 3, 'b': 4}]
float64 Int64
False True


**Hidden fact:** a NumPy array of 10 million float64 takes exactly 80 MB; the same numbers in a Python list take ~320 MB (pointers + float objects).

**Hidden fact:** pandas 3.0 (installed here) makes *copy-on-write* the default and uses a dedicated `str` dtype — chained-assignment bugs of the past now simply don't modify the original.

---
# 📝 Exam Corner

**Theory**
1. Why is NumPy faster than lists? What is vectorisation? Broadcasting?
2. Difference between `shape`, `size`, `ndim`.
3. Difference between `loc` and `iloc`.
4. How do you handle missing values in pandas?
5. Explain `groupby`, `merge` types (inner, left, right, outer).
6. Series vs DataFrame.

**Output prediction**

In [18]:
a = np.array([[1, 2, 3], [4, 5, 6]])
print(a.shape, a.T.shape, a.sum(axis=0), a.sum(axis=1))
print(a[a % 2 == 0], a[:, 1])
print(np.arange(1, 10).reshape(3, 3)[1:, :2])
s = pd.Series([10, 20, 30], index=list("abc"))
print(s["b"], s.iloc[-1], (s > 15).sum(), s.cumsum().tolist())

(2, 3) (3, 2) [5 7 9] [ 6 15]
[2 4 6] [2 5]
[[4 5]
 [7 8]]
20 30 2 [10, 30, 60]


<details><summary>💡 Answers</summary>

```
(2, 3) (3, 2) [5 7 9] [ 6 15]
[2 4 6] [2 5]
[[4 5]
 [7 8]]
20 30 2 [10, 30, 60]
```
</details>

---
## ✅ Summary
- NumPy: typed N-d arrays, vectorised maths, broadcasting, boolean masks, `axis`.
- Pandas: `DataFrame` for tables → select (`loc/iloc`, masks), clean (`fillna/dropna`), summarise (`groupby`), combine (`merge/concat`), reshape (`pivot`).
- Matplotlib for charts; always label axes and titles.

➡️ **Next:** Module 19 — Advanced Python: typing, concurrency, async, testing, packaging.